# Scouting scenarios with real observations

Illustrative, adjustable club requirements, not official club plans or budgets. Each case represents **one vacancy** and the shortlist contains alternatives. The models use frozen FBref transformations. Uploaded reported market assessments screen the ranking afterwards; they are not transfer fees, predictions, training features or verified historical information.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT/'src/config.py').exists():
    if ROOT == ROOT.parent: raise RuntimeError('Run inside the repository')
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Markdown
from src.data.provenance import sha256
report=json.loads((ROOT/'reports/scouting_scenarios/results.json').read_text())
for name, digest in report['output_sha256'].items():
    assert sha256(ROOT/'reports/scouting_scenarios'/name)==digest, name
cases=pd.DataFrame([{**case['requirements'], 'case_id':case['case_id'], 'eligible':case['eligible'], 'returned':case['returned']} for case in report['cases']])
display(cases)
display(Markdown('Dataset version: `'+report['dataset_version']+'`. No models were retrained.'))


,club,season,position,style,max_age,min_minutes,budget_million,top_n,case_id,eligible,returned
0,Valencia,2022-2023,MF,posesion,23,900,5.0,5,core:2022-2023:posesion,8,5
1,Valencia,2022-2023,DF,solidez_defensiva,27,1800,8.0,5,core:2022-2023:solidez_defensiva,10,5
2,Valencia,2022-2023,FW,vertical,25,900,10.0,5,core:2022-2023:vertical,9,5
3,Valencia,2022-2023,FW,desborde_banda,25,900,10.0,5,core:2022-2023:desborde_banda,9,5
4,Valencia,2023-2024,MF,posesion,23,900,5.0,5,core:2023-2024:posesion,2,2
5,Valencia,2023-2024,DF,solidez_defensiva,27,1800,8.0,5,core:2023-2024:solidez_defensiva,16,5
6,Valencia,2023-2024,FW,vertical,25,900,10.0,5,core:2023-2024:vertical,7,5
7,Valencia,2023-2024,FW,desborde_banda,25,900,10.0,5,core:2023-2024:desborde_banda,7,5
8,Valencia,2023-2024,FW,vertical,25,900,10.0,5,offensive:2023-2024:vertical,7,5
9,Valencia,2024-2025,FW,vertical,25,900,10.0,5,offensive:2024-2025:vertical,11,5


Dataset version: `0627515042a64840c7a63c2fef200a0cf9cf317d3105b2d7c0fdb7279f8e50c8`. No models were retrained.

## Coverage and feasibility

Core uses validation 2022/23 and held-out 2023/24 seasons; offensive uses validation 2023/24 and held-out 2024/25. These seasons follow each model’s training period. Prices use the latest reported effective date no later than June 30 of the observed season. This aligns the snapshot but does not establish that the value was published or known then. Broad historical FBref positions are retained; FW is not proof of a winger role. Players with any own-club stint are excluded. Unknown ages and unknown, zero or unmatched prices cannot pass their constraints.

In [2]:
funnel=pd.read_csv(ROOT/'reports/scouting_scenarios/funnel.csv')
display(funnel.pivot(index='case_id', columns='stage', values='count'))
for case, rows in funnel.groupby('case_id',sort=False):
    assert rows['count'].is_monotonic_decreasing
empty=int(cases.returned.eq(0).sum())
short=int(cases.returned.lt(cases.top_n).sum())
display(Markdown(f"**{empty}** cases have no feasible players; **{short}** return fewer alternatives than requested. Constraints are never relaxed automatically. Coverage exclusions describe source availability and selection bias, not poor player quality."))


stage,broad_position,exclude_all_own_club_stints,gallery,known_age_within_ceiling,known_positive_reported_value,minimum_minutes,reported_value_ceiling
case_id,,,,,,,
core:2022-2023:desborde_banda,97,32,388,42,27,35,9
core:2022-2023:posesion,134,24,388,37,18,28,8
core:2022-2023:solidez_defensiva,157,34,388,86,27,37,10
core:2022-2023:vertical,97,32,388,42,27,35,9
core:2023-2024:desborde_banda,93,25,393,30,20,27,7
core:2023-2024:posesion,138,24,393,41,22,27,2
core:2023-2024:solidez_defensiva,162,34,393,83,29,36,16
core:2023-2024:vertical,93,25,393,30,20,27,7
offensive:2023-2024:vertical,93,25,393,30,20,27,7


**0** cases have no feasible players; **1** return fewer alternatives than requested. Constraints are never relaxed automatically. Coverage exclusions describe source availability and selection bias, not poor player quality.

## Baselines and requirement sensitivity

Goals+assists per 90, goals per 90 and seeded random selection rank exactly the same eligible pool. Top-list Jaccard measures selection agreement, not recommendation relevance, statistical superiority or transfer success. Defensive and possession requirements should not be judged by goals alone. Budget and minutes vary ±20%; age varies ±2 years. Actual observed player measurements are unchanged. Empty/empty comparisons remain unknown.

In [3]:
baselines=pd.read_csv(ROOT/'reports/scouting_scenarios/baselines.csv')
sensitivity=pd.read_csv(ROOT/'reports/scouting_scenarios/sensitivity.csv')
display(baselines.groupby(['model','baseline']).shortlist_jaccard.agg(['count','mean','min','max']))
display(sensitivity.groupby(['model','variation']).agg(cases=('case_id','size'),mean_eligible=('eligible','mean'),mean_overlap=('shortlist_jaccard','mean'),min_overlap=('shortlist_jaccard','min')))
unstable=sensitivity[sensitivity.shortlist_jaccard.lt(.5)]
display(unstable)
display(Markdown(f"**{len(unstable)}** perturbed cases have Jaccard below 0.5. This is a transparent descriptive threshold, not a calibrated significance test. Low overlap means the shortlist depends strongly on the selected requirements; human review should consider alternatives near each boundary."))


count      mean       min       max
model     baseline                                                
core      goals_assists_per90      8  0.616071  0.250000  1.000000
          goals_per90              8  0.504464  0.250000  1.000000
          seeded_random_42         8  0.470238  0.250000  1.000000
offensive goals_assists_per90      2  0.428571  0.428571  0.428571
          goals_per90              2  0.547619  0.428571  0.666667
          seeded_random_42         2  0.458333  0.250000  0.666667

cases  mean_eligible  mean_overlap  min_overlap
model     variation                                                          
core      age_minus2              8          4.250      0.395833     0.285714
          age_plus2               8         12.125      0.605655     0.250000
          budget_minus20pct       8          5.750      0.568452     0.428571
          budget_plus20pct        8          9.375      0.875000     0.666667
          minutes_minus20pct      8         12.125      0.521726     0.250000
          minutes_plus20pct       8          7.250      0.732143     0.428571
offensive age_minus2              2          5.500      0.614286     0.428571
          age_plus2               2         15.500      0.214286     0.000000
          budget_minus20pct       2          8.000      0.833333     0.666667
          budget_plus20pct        2         10.500      1.000000     1.000000
          minutes_minus20pct      2         11.000      0.833333     0.666667
          minutes_plus20pct       2          8.000      0.714286     0.428571

,variation,eligible,returned,shortlist_jaccard,case_id,model,observed_season
0,budget_minus20pct,6,5,0.428571,core:2022-2023:posesion,core,2022-2023
2,age_minus2,3,3,0.333333,core:2022-2023:posesion,core,2022-2023
3,age_plus2,16,5,0.250000,core:2022-2023:posesion,core,2022-2023
4,minutes_minus20pct,13,5,0.250000,core:2022-2023:posesion,core,2022-2023
6,budget_minus20pct,6,5,0.428571,core:2022-2023:solidez_defensiva,core,2022-2023
10,minutes_minus20pct,15,5,0.428571,core:2022-2023:solidez_defensiva,core,2022-2023
12,budget_minus20pct,6,5,0.428571,core:2022-2023:vertical,core,2022-2023
14,age_minus2,3,3,0.333333,core:2022-2023:vertical,core,2022-2023
20,age_minus2,3,3,0.333333,core:2022-2023:desborde_banda,core,2022-2023
28,minutes_minus20pct,5,5,0.400000,core:2023-2024:posesion,core,2023-2024


**22** perturbed cases have Jaccard below 0.5. This is a transparent descriptive threshold, not a calibrated significance test. Low overlap means the shortlist depends strongly on the selected requirements; human review should consider alternatives near each boundary.

## Qualitative shortlist review

Inspect observed age, minutes, goals/assists and affinity alongside the source value date. The affinity is an expert-weight cosine score, not a probability or independently validated tactical fit. Remaining candidates require video, role and injury review, identity verification and actual fee negotiation. Desborde uses a broad FW proxy and measured model features; it cannot certify a winger. This local output contains provisional provider data and is excluded from Git.

In [4]:
shortlist=pd.read_csv(ROOT/'reports/scouting_scenarios/shortlist.csv')
assert not shortlist.historical_budget_eligible.any()
assert shortlist.price_identity_status.eq('provisional_not_reviewed').all()
assert shortlist.entity_id.groupby(shortlist.case_id).apply(lambda x:x.is_unique).all()
# Display aggregate observed profiles; individual provisional prices remain in the private CSV.
display(shortlist.groupby('case_id').agg(players=('entity_id','size'),min_age=('age','min'),max_age=('age','max'),min_minutes=('minutes','min'),median_affinity=('affinity','median'),median_goals_p90=('gls_p90','median'),median_assists_p90=('ast_p90','median')))
display(Markdown('These descriptive differences explain the returned profiles; they are not evidence that a club would sign or benefit from any player.'))


,players,min_age,max_age,min_minutes,median_affinity,median_goals_p90,median_assists_p90
case_id,,,,,,,
core:2022-2023:desborde_banda,5,21.0,25.0,1312.0,0.225791,0.192445,0.076923
core:2022-2023:posesion,5,21.0,23.0,1028.0,0.067605,0.087549,0.088626
core:2022-2023:solidez_defensiva,5,23.0,26.0,2156.0,0.238652,0.037625,0.000000
core:2022-2023:vertical,5,21.0,25.0,1298.0,0.600761,0.269231,0.064148
core:2023-2024:desborde_banda,5,22.0,25.0,905.0,0.493106,0.161580,0.208978
core:2023-2024:posesion,2,21.0,23.0,1790.0,-0.102937,0.097087,0.025140
core:2023-2024:solidez_defensiva,5,24.0,27.0,1963.0,0.288023,0.032131,0.045848
core:2023-2024:vertical,5,22.0,25.0,905.0,0.515565,0.161580,0.208978
offensive:2023-2024:vertical,5,22.0,24.0,905.0,0.494270,0.162553,0.054184


These descriptive differences explain the returned profiles; they are not evidence that a club would sign or benefit from any player.

## Completion and validation limits

Completed: ten observed-data cases, strict requirement funnels, same-pool baselines, sensitivity analysis, executable notebook and interactive downloadable scenarios. Partial: tactical role fit and cross-source identities remain proxies/provisional. Blocked: transfer-success validation requires reviewed identities, historically available information, defensible outcome labels and realistic fee/club requirements. No actual transfer-success score is reported. Offline execution records no human process events. The app records only explicit execution/review actions. Existing real-data model evaluation and MLflow artifacts remain unchanged; this analysis does not claim a new model experiment.